# Entrenamiento Híbrido: Extractor LungX + Vision Transformer

Este notebook está optimizado para ejecutarse en Google Colab. Integraremos la estandarización On-the-fly, el extractor de características ganador (EfficientNet-B3 + CBAM) y el modelo ViT en un solo flujo.

## 1. Configuración del Entorno y Datos

In [1]:
# Comprobación rápida del entorno
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
except Exception as e:
    print('PyTorch no está instalado o hay un error:', e)

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA disponible: True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Descomprimir el archivo directamente en el entorno local de Colab para mayor velocidad I/O
!echo "Descomprimiendo archivos..."
!unzip -q "/content/drive/MyDrive/Proyecto.zip" -d "/content/"
!echo "¡Descompresión finalizada! Los datos están listos en el disco local."

Descomprimiendo archivos...
¡Descompresión finalizada! Los datos están listos en el disco local.


In [4]:
import os
import gc
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights, vit_b_16
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Forzar la recolección de basura de Python y vaciar caché
gc.collect()
torch.cuda.empty_cache()
print("Memoria liberada. Entorno importado correctamente.")

Memoria liberada. Entorno importado correctamente.


## 2. Configuración Global (CFG)

In [ ]:
class CFG:
    seed = 42
    img_size = 384       # Tamaño base para el ExtractorLungX
    batch_size = 16      # Ajustar dependiendo si usas T4 (16-32) o A100 (64-128)
    epochs = 15
    lr = 1e-4
    zoom_factor = 1.2
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Rutas adaptadas al entorno de Colab
    TRAIN_DIR = "/content/Proyecto/data/train" 
    TEST_DIR = "/content/Proyecto/data/test" 
    
    # Ruta al modelo ganador absoluto guardado en el experimento anterior
    MODEL_PATH = "/content/drive/MyDrive/Proyecto/resultados_lungx/mejor_modelo_absoluto.pth"
    
    # Directorio para guardar resultados
    SAVE_DIR = "/content/drive/MyDrive/Proyecto/resultados_lungx/"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print(f"Device => {CFG.device}")

Device => cuda


## 3. Bloques de la Arquitectura Ganadora (Extractor LungX)

In [ ]:
class BackboneEfficientNetB3(nn.Module):
    def __init__(self):
        super().__init__()
        modelo_base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.features = modelo_base.features
        self.indices_extraccion = [3, 5, 7]

    def forward(self, x):
        mapas_multiescala = []
        for i, capa in enumerate(self.features):
            x = capa(x)
            if i in self.indices_extraccion:
                mapas_multiescala.append(x)
            if i == max(self.indices_extraccion):
                break
        return mapas_multiescala

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.channel_attention(x)
        return out * self.spatial_attention(out)

class MultiScaleFusionBlock(nn.Module):
    def __init__(self, in_channels_list, out_channels=384, target_size=(19, 19)):
        super(MultiScaleFusionBlock, self).__init__()
        self.projections = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1) 
            for in_channels in in_channels_list
        ])
        self.target_size = target_size

    def forward(self, mapas_cbam):
        fused_map = 0
        for i, mapa in enumerate(mapas_cbam):
            mapa_proyectado = self.projections[i](mapa)
            mapa_redimensionado = F.interpolate(
                mapa_proyectado, size=self.target_size, mode='bilinear', align_corners=False
            )
            fused_map = fused_map + mapa_redimensionado
        return fused_map

class ExtractorLungX(nn.Module):
    def __init__(self):
        super(ExtractorLungX, self).__init__()
        self.backbone = BackboneEfficientNetB3()
        canales = [48, 136, 384]
        self.cbam_modules = nn.ModuleList([CBAM(c) for c in canales])
        self.fusion = MultiScaleFusionBlock(canales)

    def forward(self, x):
        mapas = self.backbone(x)
        mapas_listos = [cbam(mapa) for cbam, mapa in zip(self.cbam_modules, mapas)]
        return self.fusion(mapas_listos)

## 4. Fusión Híbrida: Extractor + Vision Transformer

In [7]:
class HybridViTPneumonia(nn.Module):
    def __init__(self, extractor_weights_path=None):
        super().__init__()
        
        # 1. Extractor de características preentrenado
        self.extractor = ExtractorLungX()
        if extractor_weights_path and os.path.exists(extractor_weights_path):
            print(f"[INFO] Cargando pesos de ExtractorLungX desde: {extractor_weights_path}")
            self.extractor.load_state_dict(torch.load(extractor_weights_path, map_location='cpu'), strict=False)
        else:
            print("[WARN] No se encontraron pesos previos. Entrenando extractor desde cero.")
        
        # 2. Vision Transformer
        vit = vit_b_16(weights="IMAGENET1K_V1")
        
        # Modificamos el Patch Embedding para aceptar los mapas del extractor (384 canales)
        self.patch_embedding = nn.Conv2d(in_channels=384, out_channels=vit.hidden_dim, kernel_size=1)
        
        # CORRECCIÓN 1: En PyTorch/Torchvision se llama 'class_token', no 'cls_token'
        self.cls_token = vit.class_token
        
        # CORRECCIÓN 2: El encoder de ViT ya incluye pos_embedding y LayerNorm interno.
        # Interpolamos a la nueva longitud (19x19 + 1 = 362) y sobrescribimos el parámetro
        pos_emb = vit.encoder.pos_embedding # [1, 197, 768]
        new_pos_emb = F.interpolate(
            pos_emb.transpose(1, 2), 
            size=362, 
            mode='linear', 
            align_corners=False
        ).transpose(1, 2)
        
        vit.encoder.pos_embedding = nn.Parameter(new_pos_emb)
        self.vit_encoder = vit.encoder
        
        # Guardamos la cabeza de clasificación final
        self.head = nn.Linear(vit.hidden_dim, 1)

    def forward(self, x):
        # Mapa multiescala [Batch, 384, 19, 19]
        features = self.extractor(x)
        
        # Proyección y aplanamiento a tokens
        tokens = self.patch_embedding(features) # [B, 768, 19, 19]
        tokens = tokens.flatten(2).transpose(1, 2) # [B, 361, 768]
        
        # Concatenar CLS token
        batch_size = tokens.shape[0]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat((cls_tokens, tokens), dim=1) # [B, 362, 768]
        
        # Transformer Encoder (Ya suma el pos_embedding y aplica LayerNorm internamente)
        features_encoded = self.vit_encoder(tokens)
        
        # Clasificación a partir del CLS Token (índice 0)
        return self.head(features_encoded[:, 0])

## 5. Pipeline de Datos (Estandarización On-The-Fly)

In [8]:
def build_transforms():
    train_transforms = T.Compose([
        T.RandomHorizontalFlip(0.5),
        T.ToTensor(),
        T.Lambda(lambda x: x.repeat(3, 1, 1)),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    val_transforms = T.Compose([
        T.ToTensor(),
        T.Lambda(lambda x: x.repeat(3, 1, 1)),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    return train_transforms, val_transforms

class OnlineStandardizedXRayDataset(Dataset):
    def __init__(self, dataframe, transforms):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = str(row["path"])
        
        # Carga OpenCV en Grises
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")
            
        # Crop Cuadrado
        h, w = img.shape[:2]
        min_dim = min(h, w)
        start_x = (w - min_dim) // 2
        start_y = (h - min_dim) // 2
        square_img = img[start_y:start_y+min_dim, start_x:start_x+min_dim]
        
        # Zoom In
        zoom_dim = int(min_dim / CFG.zoom_factor)
        zoom_start = (min_dim - zoom_dim) // 2
        zoomed_img = square_img[zoom_start : zoom_start + zoom_dim, zoom_start : zoom_start + zoom_dim]
        
        # Resize a img_size (384)
        resized_img = cv2.resize(zoomed_img, (CFG.img_size, CFG.img_size), interpolation=cv2.INTER_CUBIC)
        
        # CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        standardized_img = clahe.apply(resized_img)
        
        # A PIL para Torchvision
        image = Image.fromarray(standardized_img)
        image = self.transforms(image)
        
        label = torch.tensor(row["label"]).float()
        return image, label

def build_dataframe():
    rows = []
    normal_dir = Path(CFG.TRAIN_DIR) / "NORMAL"
    pneumonia_dir = Path(CFG.TRAIN_DIR) / "PNEUMONIA"

    if not normal_dir.exists():
        print(f"⚠️ ADVERTENCIA: No se encuentra la carpeta NORMAL en {normal_dir}")
        print("Por favor verifica que la extracción del ZIP fue exitosa.")

    for img_path in normal_dir.rglob("*"):
        if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
            rows.append({"path": str(img_path), "label": 0})

    for img_path in pneumonia_dir.rglob("*"):
        if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
            rows.append({"path": str(img_path), "label": 1})

    return pd.DataFrame(rows)

## 6. Motor de Entrenamiento y Evaluación

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0
    for images, labels in loader:
        images, labels = images.to(CFG.device), labels.to(CFG.device)
      
        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

def validate(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(CFG.device)
            outputs = model(images).squeeze(1)
            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)
    preds = (all_probs > 0.5).astype(int)
    return f1_score(all_targets, preds)

## 7. Ejecución Principal y Respaldo a Google Drive

In [ ]:
def run_training():
    df = build_dataframe()
    print(f"Se encontraron {len(df)} imágenes en total.")
    
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=CFG.seed, stratify=df["label"])

    train_tfms, val_tfms = build_transforms()
    train_ds = OnlineStandardizedXRayDataset(train_df, train_tfms)
    val_ds = OnlineStandardizedXRayDataset(val_df, val_tfms)

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = HybridViTPneumonia(extractor_weights_path=CFG.MODEL_PATH).to(CFG.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0.0
    local_model_name = "vit_pneumonia_hybrid.pth"
    
    for epoch in range(CFG.epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_f1 = validate(model, val_loader)
        
        print(f"Epoch {epoch+1:02d}/{CFG.epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), local_model_name)
            print("  => ¡Mejor F1 alcanzado! Modelo guardado temporalmente.")

    print("\n🏆 Entrenamiento Completado 🏆")
    
    # --- Respaldo automático a Google Drive ---
    os.makedirs(CFG.SAVE_DIR, exist_ok=True)
    print(f"Copiando archivo a Drive: {CFG.SAVE_DIR} ...")
    
    if os.path.exists(local_model_name):
        destino_final = os.path.join(CFG.SAVE_DIR, local_model_name)
        shutil.copy(local_model_name, destino_final)
        print(f"✅ Respaldo exitoso: {destino_final}")
    else:
        print("❌ No se encontró el archivo del modelo para copiar.")

if __name__ == "__main__":
    run_training()

Se encontraron 5232 imágenes en total.
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 231MB/s]


[INFO] Cargando pesos de ExtractorLungX desde: /content/drive/MyDrive/Proyecto/resultados_lungx/mejor_modelo_absoluto.pth
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 246MB/s] 


## 8. Inferencia en el Test Set y Submit para Kaggle

Con esta sección leemos las imágenes de prueba sin etiquetas, realizamos las predicciones con el modelo recién entrenado y guardamos un archivo `submission.csv` listo para subir a la competencia.

In [ ]:
# 1. Dataset específico para Kaggle Test (sin etiquetas, solo retorna imagen y nombre de archivo)
class KaggleTestDataset(Dataset):
    def __init__(self, img_paths, transforms):
        self.img_paths = img_paths
        self.transforms = transforms

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = str(self.img_paths[idx])
        filename = os.path.basename(img_path)

        # Mismo preprocesamiento On-the-fly que en entrenamiento
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"No se pudo leer: {img_path}")
            
        h, w = img.shape[:2]
        min_dim = min(h, w)
        start_x = (w - min_dim) // 2
        start_y = (h - min_dim) // 2
        square_img = img[start_y:start_y+min_dim, start_x:start_x+min_dim]
        
        zoom_dim = int(min_dim / CFG.zoom_factor)
        zoom_start = (min_dim - zoom_dim) // 2
        zoomed_img = square_img[zoom_start : zoom_start + zoom_dim, zoom_start : zoom_start + zoom_dim]
        
        resized_img = cv2.resize(zoomed_img, (CFG.img_size, CFG.img_size), interpolation=cv2.INTER_CUBIC)
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        standardized_img = clahe.apply(resized_img)
        
        image = Image.fromarray(standardized_img)
        image = self.transforms(image)
        
        return image, filename

def generate_kaggle_submission():
    print("\n" + "="*50)
    print("Preparando generación del archivo para Kaggle...")
    test_dir = Path(CFG.TEST_DIR)
    
    if not test_dir.exists():
        print(f"⚠️ No se encontró la carpeta test en {test_dir}.")
        print("Asegúrate de que la ruta sea correcta.")
        return

    # Buscar todas las imágenes en test
    test_images = [p for p in test_dir.rglob('*') if p.suffix.lower() in ['.png', '.jpg', '.jpeg']]
    print(f"Se encontraron {len(test_images)} imágenes de prueba.")

    # Usar las transformaciones de validación (sin data augmentation, solo normalización)
    _, val_tfms = build_transforms()
    test_ds = KaggleTestDataset(test_images, val_tfms)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Cargar el modelo que acabamos de entrenar
    model = HybridViTPneumonia().to(CFG.device)
    local_model_name = "vit_pneumonia_hybrid.pth"
    
    try:
        model.load_state_dict(torch.load(local_model_name, map_location=CFG.device))
        print("✅ Pesos del modelo híbrido cargados correctamente.")
    except FileNotFoundError:
        print("⚠️ No se encontró el modelo entrenado. Ejecuta el entrenamiento de arriba primero.")
        return

    model.eval()
    predictions = []
    filenames = []

    print("Ejecutando inferencia en el Test Set...")
    with torch.no_grad():
        for images, fnames in test_loader:
            images = images.to(CFG.device)
            outputs = model(images).squeeze(1)
            probs = torch.sigmoid(outputs)
            
            # Convertir probabilidad a clase binaria (0 o 1)
            preds = (probs > 0.5).int().cpu().numpy()
            
            predictions.extend(preds)
            filenames.extend(fnames)

    # Construir DataFrame. 
    # NOTA: Cambia los nombres 'Id' y 'Label' si Kaggle exige columnas con nombres distintos.
    submission_df = pd.DataFrame({
        'Id': filenames,
        'Label': predictions
    })
    
    sub_path = "submission.csv"
    submission_df.to_csv(sub_path, index=False)
    print(f"\n✅ ¡Submission generada con éxito! Guardada en local como: {sub_path}")
    print("\nVista previa del archivo:")
    print(submission_df.head())

    # Respaldo automático en Google Drive para que no se pierda
    destino_drive = os.path.join(CFG.SAVE_DIR, "submission.csv")
    shutil.copy(sub_path, destino_drive)
    print(f"✅ Copia de seguridad guardada en tu Drive: {destino_drive}")
    print("="*50)

# Descomenta la siguiente línea para generar el CSV al correr la celda
# generate_kaggle_submission()

: 